In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("正在加载第一周处理好的带有时间特征的数据...")
# 使用绝对路径读取
base_path = '/Users/huyujie/Documents/amazon-supply-chain-project/data/processed/'
train_amazon = pd.read_csv(base_path + 'train_amazon_features.csv')
train_uci = pd.read_csv(base_path + 'train_uci_features.csv')

print("✅ 数据加载成功！")

正在加载第一周处理好的带有时间特征的数据...
✅ 数据加载成功！


In [2]:
print(f"清理前 UCI数据行数: {len(train_uci)}")

# 删除完全重复的行
train_uci = train_uci.drop_duplicates().reset_index(drop=True)

print(f"清理后 UCI数据行数: {len(train_uci)}")
print("✅ 重复值清理完成！")

清理前 UCI数据行数: 431513
清理后 UCI数据行数: 427878
✅ 重复值清理完成！


In [3]:
# 1. 汇总 Amazon 数据集
# 按照 日期(OrderDate) 和 商品ID(ProductID) 分组
amazon_daily = train_amazon.groupby(['OrderDate', 'ProductID']).agg({
    'Quantity': 'sum',        # 销量相加
    'TotalAmount': 'sum',     # 销售额相加
    'UnitPrice': 'mean',      # 单价取平均
    'Category': 'first',      # 类别保留原样
    'IsHoliday': 'first'      # 节假日标签保留
}).reset_index()

# 2. 汇总 UCI 数据集
# 按照 日期(InvoiceDate) 和 商品代码(StockCode) 分组
# 注意：UCI的时间带有时分秒，我们需要先把它变成纯日期(年月日)
train_uci['JustDate'] = pd.to_datetime(train_uci['InvoiceDate']).dt.date
uci_daily = train_uci.groupby(['JustDate', 'StockCode']).agg({
    'Quantity': 'sum',
    'UnitPrice': 'mean',
    'Description': 'first',
    'IsHoliday': 'first'
}).reset_index()

print("✅ 产品-日期粒度的销售数据集 构建完成！")
print("Amazon汇总后前3行预览：")
display(amazon_daily.head(3))

✅ 产品-日期粒度的销售数据集 构建完成！
Amazon汇总后前3行预览：


,OrderDate,ProductID,Quantity,TotalAmount,UnitPrice,Category,IsHoliday
0,2020-01-01,P00001,2,1066.43,530.010,Toys & Games,1
1,2020-01-01,P00002,5,34.02,6.760,Home & Kitchen,1
2,2020-01-01,P00003,4,1891.58,424.045,Sports & Outdoors,1


In [4]:
# 保存到 processed 文件夹
amazon_daily.to_csv(base_path + 'amazon_daily_sales_train.csv', index=False)
uci_daily.to_csv(base_path + 'uci_daily_sales_train.csv', index=False)

print("✅ 第二周交付物：处理后的干净数据集(CSV格式) 已成功保存！")

✅ 第二周交付物：处理后的干净数据集(CSV格式) 已成功保存！
